<a href="https://www.kaggle.com/code/avikdas567/wids-global-datathon-2026?scriptVersionId=306935472" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata
import warnings
warnings.filterwarnings("ignore")

# Paths

TRAIN_PATH = "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/train.csv"
TEST_PATH = "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/test.csv"

# Load

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

ID_COL = "event_id"
TIME_COL = "time_to_hit_hours"
EVENT_COL = "event"

FEATURES = [c for c in train.columns if c not in [ID_COL, TIME_COL, EVENT_COL]]

# Preprocessing

for col in FEATURES:
    if train[col].dtype == "object":
        train[col] = train[col].fillna("missing")
        test[col] = test[col].fillna("missing")
    else:
        med = train[col].median()
        train[col] = train[col].fillna(med)
        test[col] = test[col].fillna(med)

# Feature scaling

scaler = StandardScaler()
train[FEATURES] = scaler.fit_transform(train[FEATURES])
test[FEATURES] = scaler.transform(test[FEATURES])

# Horizons

HORIZONS = [12, 24, 48, 72]

# Target builder

def build_target(df, horizon):
    y, mask = [], []
    for t, e in zip(df[TIME_COL], df[EVENT_COL]):
        if e == 1 and t <= horizon:
            y.append(1); mask.append(True)
        elif t > horizon:
            y.append(0); mask.append(True)
        else:
            y.append(0); mask.append(False)
    return np.array(y), np.array(mask)

# Noise injection

def add_noise(X, noise_level=0.01):
    return X + noise_level * np.random.randn(*X.shape)

# Params

params_cls = {
    "objective": "binary",
    "learning_rate": 0.02,
    "num_leaves": 16,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.7,
    "bagging_freq": 5,
    "min_data_in_leaf": 10,
    "verbosity": -1,
    "seed": 42
}

params_reg = {
    "objective": "regression",
    "learning_rate": 0.02,
    "num_leaves": 16,
    "feature_fraction": 0.7,
    "min_data_in_leaf": 10,
    "verbosity": -1,
    "seed": 42
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# STORAGE

oof_cls = {h: np.zeros(len(train)) for h in HORIZONS}
test_cls = {h: np.zeros(len(test)) for h in HORIZONS}

oof_reg = np.zeros(len(train))
test_reg = np.zeros(len(test))

oof_rank = np.zeros(len(train))
test_rank = np.zeros(len(test))

# Classification models

for h in HORIZONS:
    print(f"Training CLS {h}h")

    y, mask = build_target(train, h)
    X = train[FEATURES][mask]
    y = y[mask]

    oof = np.zeros(len(X))
    preds = np.zeros(len(test))

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr = add_noise(X.iloc[tr_idx].values)
        y_tr = y[tr_idx]

        X_val = X.iloc[val_idx].values
        y_val = y[val_idx]

        model = lgb.LGBMClassifier(**params_cls, n_estimators=3000)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(200, verbose=False)]
        )

        oof[val_idx] = model.predict_proba(X_val)[:,1]
        preds += model.predict_proba(test[FEATURES])[:,1] / kf.n_splits

    oof_cls[h][mask] = oof
    test_cls[h] = preds

# Survival regression

print("Training REG")

X = train[FEATURES]
y = train[TIME_COL]

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr = add_noise(X.iloc[tr_idx].values)
    y_tr = y.iloc[tr_idx]

    X_val = X.iloc[val_idx].values
    y_val = y.iloc[val_idx]

    model = lgb.LGBMRegressor(**params_reg, n_estimators=3000)

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(200, verbose=False)]
    )

    oof_reg[val_idx] = model.predict(X_val)
    test_reg += model.predict(test[FEATURES]) / kf.n_splits

def reg_to_prob(pred_time, h):
    return 1 / (1 + np.exp((pred_time - h)/8))

reg_probs = {h: reg_to_prob(test_reg, h) for h in HORIZONS}

# Ranking model

print("Training RANK")

y_rank = -train[TIME_COL]

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    model = lgb.LGBMRegressor(**params_reg, n_estimators=3000)

    model.fit(
        train.iloc[tr_idx][FEATURES],
        y_rank.iloc[tr_idx],
        eval_set=[(train.iloc[val_idx][FEATURES], y_rank.iloc[val_idx])],
        callbacks=[lgb.early_stopping(200, verbose=False)]
    )

    oof_rank[val_idx] = model.predict(train.iloc[val_idx][FEATURES])
    test_rank += model.predict(test[FEATURES]) / kf.n_splits

test_rank = rankdata(test_rank) / len(test_rank)

# META MODEL (stacking)

print("Training META")

meta_train = []
meta_test = []

for h in HORIZONS:
    meta_train.append(oof_cls[h])
    meta_test.append(test_cls[h])

meta_train.append(oof_reg)
meta_test.append(test_reg)

meta_train.append(oof_rank)
meta_test.append(test_rank)

meta_train = np.vstack(meta_train).T
meta_test = np.vstack(meta_test).T

meta_preds = {}

for i, h in enumerate(HORIZONS):
    y, mask = build_target(train, h)

    X_meta = meta_train[mask]
    y_meta = y[mask]

    preds = np.zeros(len(test))

    for tr_idx, val_idx in kf.split(X_meta):
        model = lgb.LGBMClassifier(
            objective="binary",
            learning_rate=0.02,
            num_leaves=8,
            n_estimators=2000,
            verbosity=-1
        )

        model.fit(
            X_meta[tr_idx], y_meta[tr_idx],
            eval_set=[(X_meta[val_idx], y_meta[val_idx])],
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )

        preds += model.predict_proba(meta_test)[:,1] / kf.n_splits

    meta_preds[h] = preds

# Final blending

final = {}

for h in HORIZONS:
    base = (
        0.4 * test_cls[h] +
        0.3 * reg_probs[h] +
        0.3 * test_rank
    )

    final[h] = 0.6 * meta_preds[h] + 0.4 * base

# Calibration

for h in HORIZONS:
    r = rankdata(final[h]) / len(final[h])
    final[h] = 0.75 * final[h] + 0.25 * r

# Submission

submission = pd.DataFrame({
    "event_id": test[ID_COL],
    "prob_12h": final[12],
    "prob_24h": final[24],
    "prob_48h": final[48],
    "prob_72h": final[72],
})

# Monotonic enforcement

submission["prob_24h"] = np.maximum(submission["prob_24h"], submission["prob_12h"])
submission["prob_48h"] = np.maximum(submission["prob_48h"], submission["prob_24h"])
submission["prob_72h"] = np.maximum(submission["prob_72h"], submission["prob_48h"])

for col in ["prob_12h","prob_24h","prob_48h","prob_72h"]:
    submission[col] = submission[col].clip(0,1)

# Save

submission.to_csv("submission.csv", index=False)

print("\n'submission.csv' created.\n")
display(submission.head())

Training CLS 12h
Training CLS 24h
Training CLS 48h
Training CLS 72h
Training REG
Training RANK
Training META

'submission.csv' created.



,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.410210,0.638781,0.638781,0.638781
1,13353600,0.735075,0.901571,0.946616,0.946616
2,13942327,0.318255,0.547970,0.547970,0.547970
3,16112781,0.741332,0.905346,0.952018,0.952018
4,17132808,0.426601,0.671404,0.691755,0.691755
